# SAGE reproduction on Google Colab

Runtime → Change runtime type → **T4 GPU** (free) or better. Outputs are saved to Google Drive, so a disconnect loses nothing: re-run the setup cells and continue (the LLM baseline resumes where it stopped).

Times below are rough estimates for a T4.

In [ ]:
!nvidia-smi

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/drive/MyDrive/Sage'
if not os.path.exists(REPO):
    !git clone https://github.com/Floxy1991/Sage.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!pip -q install bitsandbytes rank-bm25 sentencepiece accelerate

In [ ]:
# Optional: only needed for the gated LLaMA-3.1-8B baseline (your HF account must have access).
# from huggingface_hub import notebook_login; notebook_login()

## 2. Quick checks (200 examples each, ~15 min total)

In [ ]:
!python llm_baseline.py --model Qwen/Qwen2.5-7B-Instruct --split paper --limit 200 --load-4bit --out results/qwen_check

In [ ]:
!python sage.py train-selfret --epochs 1 --out ckpt/selfret_check
!python sage.py build-index --model ckpt/selfret_check --out ckpt/index_check
!python sage.py train-fusion --epochs 1 --selfret ckpt/selfret_check --index ckpt/index_check --knowledge none --out ckpt/fusion_check
!python sage.py infer --limit 200 --selfret ckpt/selfret_check --index ckpt/index_check --fusion ckpt/fusion_check --knowledge none --out results/sage_check

## 3. LLM baselines, full test set (~1-2 h each on T4)

In [ ]:
!python llm_baseline.py --model Qwen/Qwen2.5-7B-Instruct --split paper --load-4bit --out results/qwen

In [ ]:
!python llm_baseline.py --model mistralai/Mistral-7B-Instruct-v0.3 --split paper --load-4bit --out results/mistral

In [ ]:
# Needs HF login with LLaMA access (cell in section 1).
# !python llm_baseline.py --model meta-llama/Llama-3.1-8B-Instruct --split paper --load-4bit --out results/llama

## 4. SAGE re-implementation, full (~3-5 h on T4)

In [ ]:
!python sage.py train-selfret --epochs 3
!python sage.py build-index
!python sage.py train-fusion --epochs 3 --knowledge none

In [ ]:
!python sage.py infer --knowledge none --out results/sage
!python sage.py eval-retrieval --out results/sage

## 5. Ablations (Table 6), optional

In [ ]:
for flag in ['no-constraint', 'no-resid', 'retrieval-only']:
    !python sage.py infer --knowledge none --{flag} --out results/sage_{flag}

## 6. Compare with the paper

In [ ]:
!python compare.py | tee results/comparison.md